# 01 — Data preparation (C4)

Beginner path: install packages → get C4 source into the runtime → load the **real** ASSISTments corrected CSV → quality report → learner splits → fit response model on **train only**.

**Expected outputs** (under your output/Drive folder):
- `artifacts/quality_report.json`
- `artifacts/real_data_provenance.json`
- `artifacts/split_meta.json`
- `artifacts/response_model.joblib`
- `artifacts/skill_catalogue.json`
- feature pickle files

This notebook does **not** train RL policies.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 0 — Configuration

Edit these values before running other cells.


In [6]:
RUN_IN_COLAB = True
DRIVE_SUBDIR = "AdaptiveLearnSE/c4_runs"
ALLOW_GDOWN = False

CSV_PATH = (
    "/content/drive/MyDrive/AdaptiveLearnSE/"
    "c4_data/raw/skill_builder_data_corrected.csv"
)

TOP_K_SKILLS = 20
SEED = 42
SOURCE_MODE = "zip"

## Step 1 — Install training dependencies (run before project imports)


In [ ]:
import sys
print(sys.executable, sys.version)


/usr/bin/python3 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## Step 2 — Obtain C4 source (ZIP recommended for unpushed work)


In [7]:
from pathlib import Path, PurePosixPath
import zipfile
import tempfile
import shutil
import sys
import subprocess

zip_path = Path("/content/c4-source.zip")

if not zip_path.is_file():
    from google.colab import files

    print("Choose c4-source.zip")
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".zip")]

    if len(names) != 1:
        raise ValueError("Select exactly one source ZIP.")

    zip_path = Path(names[0]).resolve()

dest = Path(tempfile.mkdtemp(prefix="c4_source_", dir="/content"))

with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        name = member.filename.replace("\\", "/")
        relative = PurePosixPath(name)

        if (
            relative.is_absolute()
            or ".." in relative.parts
            or any(":" in part for part in relative.parts)
        ):
            raise ValueError(f"Unsafe ZIP path: {name}")

        target = dest.joinpath(*relative.parts)

        if member.is_dir() or name.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)

roots = [
    p.parent
    for p in dest.rglob("requirements-training.txt")
    if (p.parent / "training").is_dir()
]

if len(roots) != 1:
    raise RuntimeError(f"Expected one C4 folder; found {len(roots)}.")

C4_ROOT = roots[0]
sys.path.insert(0, str(C4_ROOT))

print("C4_ROOT:", C4_ROOT)

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-r", str(C4_ROOT / "requirements-training.txt")
])

print("C4 source and dependencies ready.")

C4_ROOT: /content/c4_source_jjy6gs68/c4
C4 source and dependencies ready.


## Step 3 — Output directories (Drive if available)


In [10]:
from google.colab import drive
from pathlib import Path
from training.paths import stage_dirs

drive.mount("/content/drive")

MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not mounted. Complete authorization.")

OUT = MY_DRIVE / DRIVE_SUBDIR
OUT.mkdir(parents=True, exist_ok=True)

DIRS = stage_dirs(OUT)
ARTIFACTS = DIRS["artifacts"]

print("Output root:", OUT)
print("Artifacts:", ARTIFACTS)

Mounted at /content/drive
Output root: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs
Artifacts: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/artifacts


## Step 4 — Acquire ASSISTments corrected CSV

Official page: https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010

Corrected Drive link from that page: https://drive.google.com/file/d/1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE/view?usp=sharing

If you need to upload manually in Colab, run the next cell.


In [11]:
%pip -q install gdown

from pathlib import Path
import gdown
import pandas as pd

dataset = Path(CSV_PATH)
dataset.parent.mkdir(parents=True, exist_ok=True)

def check_csv(path):
    with path.open("rb") as f:
        header = f.read(4096)

    if header.startswith(b"PK"):
        raise ValueError("Downloaded file is ZIP, not CSV.")

    if b"<html" in header.lower() or b"<!doctype html" in header.lower():
        raise ValueError("Downloaded file is an HTML error page, not CSV.")

    # Use an encoding fallback only if UTF-8 decoding fails.
    try:
        sample = pd.read_csv(path, nrows=5)
    except UnicodeDecodeError:
        sample = pd.read_csv(path, nrows=5, encoding="latin-1")

    required = {"user_id", "problem_id", "skill_id", "correct"}
    missing = required - set(sample.columns)

    if missing:
        raise ValueError(f"Expected columns missing: {sorted(missing)}")

    return sample

if dataset.is_file() and dataset.stat().st_size > 0:
    print("Using existing Drive dataset.")
    preview = check_csv(dataset)
else:
    partial = dataset.with_suffix(".download")

    result = gdown.download(
        id="1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE",
        output=str(partial),
        quiet=False,
    )

    if result is None or not partial.is_file():
        raise RuntimeError("Download failed. Check permission/quota messages.")

    if partial.stat().st_size == 0:
        raise RuntimeError("Downloaded file is empty.")

    preview = check_csv(partial)
    partial.replace(dataset)

print("Dataset path:", dataset)
print("Size MB:", round(dataset.stat().st_size / (1024 ** 2), 2))
print("Columns:", preview.columns.tolist())
print("Initial CSV checks passed.")

Using existing Drive dataset.
Dataset path: /content/drive/MyDrive/AdaptiveLearnSE/c4_data/raw/skill_builder_data_corrected.csv
Size MB: 61.43
Columns: ['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']
Initial CSV checks passed.


## Step 5 — Run data preparation


In [12]:
from training.pipeline import run_data_preparation
from training.config import SplitConfig

result = run_data_preparation(
    csv_path=CSV_PATH,
    output_dir=ARTIFACTS,
    split_config=SplitConfig(seed=SEED),
    allow_download=ALLOW_GDOWN,
    top_k_skills=TOP_K_SKILLS,
    seed=SEED,
)

print("Done. Keys:", sorted(result.keys()))
print("Learners:", result["split_meta"]["n_learners"])
print("Response val metrics:", result["response_metrics"]["validation"])
print("IMPORTANT: test metrics are held out — do not retune on them.")
print(result["response_metrics"]["test_held_out"])

Done. Keys: ['csv_path', 'output_dir', 'provenance', 'quality_report', 'response_metrics', 'skill_catalogue', 'split_meta']
Learners: {'train': 2952, 'validation': 633, 'test': 632}
Response val metrics: {'accuracy': 0.6512508057975837, 'brier': 0.2188888690521609, 'log_loss': 0.6278943163132447, 'n': 48089, 'positive_rate': 0.6350516750192352, 'roc_auc': 0.7067392561134989}
IMPORTANT: test metrics are held out — do not retune on them.
{'accuracy': 0.6662353520074357, 'brier': 0.2156912730749588, 'log_loss': 0.6215043051409034, 'n': 54871, 'positive_rate': 0.640720963714895, 'roc_auc': 0.7119660044977623}


## Step 6 — Quick peek at the quality report


In [13]:
import json

qr = json.loads((ARTIFACTS / "quality_report.json").read_text())

print("columns:", len(qr["available_columns"]))
print("learners:", qr["n_learners"])
print("interactions:", qr["n_normalised_interactions"])
print("multi-skill fraction:", qr["multi_skill"]["fraction"])
print("domain_warning:", qr["domain_warning"])

columns: 31
learners: 4217
interactions: 346860
multi-skill fraction: 0.1356080262930289


In [14]:
print("Saved artifacts:")
for p in sorted(Path(ARTIFACTS).iterdir()):
    if p.is_file():
        print(p.name)

print("\nSplit summary:")
split_path = Path(ARTIFACTS) / "split_meta.json"
print(split_path.read_text())

Saved artifacts:
features_test.pkl
features_train.pkl
features_validation.pkl
interactions_test.pkl
interactions_train.pkl
interactions_validation.pkl
quality_report.json
real_data_provenance.json
response_metrics.json
response_model.joblib
skill_catalogue.json
split_meta.json

Split summary:
{
  "actual_test_learners": 632,
  "fractions": {
    "test": 0.15,
    "train": 0.7,
    "validation": 0.15
  },
  "n_interactions": {
    "test": 54871,
    "train": 243900,
    "validation": 48089
  },
  "n_learners": {
    "test": 632,
    "train": 2952,
    "validation": 633
  },
  "seed": 42
}


In [15]:
from pathlib import Path
from training.paths import stage_dirs
from training.config import EnvConfig
from training.pipeline import (
    run_simulator_stage,
    load_simulator_from_artifacts,
)
from training.environment import make_env
from gymnasium.utils.env_checker import check_env

OUT = Path("/content/drive/MyDrive/AdaptiveLearnSE/c4_runs")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted.")

DIRS = stage_dirs(OUT)
ARTIFACTS = DIRS["artifacts"]

required = [
    "response_model.joblib",
    "skill_catalogue.json",
    "features_train.pkl",
    "features_validation.pkl",
    "features_test.pkl",
]

missing = [
    name for name in required
    if not (ARTIFACTS / name).is_file()
]

if missing:
    raise FileNotFoundError(f"Missing Notebook 01 artifacts: {missing}")

env_cfg = EnvConfig(include_simulation_only_actions=False)

sim_info = run_simulator_stage(
    ARTIFACTS,
    output_dir=DIRS["synthetic"],
    env_config=env_cfg,
    seed=42,
)

print("Simulator stage completed:")
print(sim_info)

Simulator stage completed:
{'sim_output_dir': '/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/synthetic', 'trajectory_paths': {'sim_train': '/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/synthetic/sim_train_trajectories.csv', 'sim_val': '/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/synthetic/sim_val_trajectories.csv', 'sim_test': '/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/synthetic/sim_test_trajectories.csv'}}


In [17]:
from pathlib import Path
import importlib
import training.environment as env_module

# Locate the actual imported source file.
env_file = Path(env_module.__file__)
source = env_file.read_text(encoding="utf-8")

old = """    def reset(self, *, seed: int | None = None, options: dict | None = None):
        if seed is not None:
            self._rng = np.random.default_rng(seed)
"""

new = """    def reset(self, *, seed: int | None = None, options: dict | None = None):
        super().reset(seed=seed)
        self._rng = self.np_random
"""

if old in source:
    backup = env_file.with_suffix(".py.before_seed_fix")
    if not backup.exists():
        backup.write_text(source, encoding="utf-8")

    env_file.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("Seed initialization fixed:", env_file)

elif new in source:
    print("Fix already applied.")

else:
    raise RuntimeError(
        "Source differs from the inspected version. "
        "Do not apply an automatic replacement."
    )

importlib.invalidate_caches()
env_module = importlib.reload(env_module)
make_env = env_module.make_env

print("Updated environment module loaded.")

Seed initialization fixed: /content/c4_source_cgeqphp4/c4/training/environment.py
Updated environment module loaded.


In [18]:
sim = load_simulator_from_artifacts(ARTIFACTS, env_cfg)
env = make_env(sim, env_cfg)

check_env(env, skip_render_check=True)

obs, info = env.reset(seed=0)

print("Environment checks passed.")
print("Observation shape:", obs.shape)
print("Number of actions:", env.action_space.n)
print("Action examples:", env.action_catalogue()[:3])

assumptions = sim.assumptions()
print("Fitted from data:", assumptions["fitted_from_data"])
print("Assumed parameters:", list(assumptions["assumed_not_fitted"]))

print("\nSaved synthetic files:")
for p in sorted(DIRS["synthetic"].iterdir()):
    if p.is_file():
        print(p.name)

env.close()

Environment checks passed.
Observation shape: (24,)
Number of actions: 20
Action examples: [{'action_id': 0, 'label': '311', 'kind': 'assistments_skill_practice', 'real_assistments_support': True}, {'action_id': 1, 'label': '47', 'kind': 'assistments_skill_practice', 'real_assistments_support': True}, {'action_id': 2, 'label': '277', 'kind': 'assistments_skill_practice', 'real_assistments_support': True}]
Fitted from data: ['logistic_regression_response_coefficients', 'feature_scaler_means_stds', 'skill_catalogue_from_train_frequency']
Assumed parameters: ['assumed_mastery_gain_correct', 'assumed_mastery_gain_incorrect', 'assumed_forgetting_per_step', 'blend_response_model_with_mastery', 'initial_mastery_sampling']

Saved synthetic files:
SYNTHETIC_README.json
action_catalogue.json
sim_test_provenance.json
sim_test_trajectories.csv
sim_train_provenance.json
sim_train_trajectories.csv
sim_val_provenance.json
sim_val_trajectories.csv
simulator_assumptions.json


In [19]:
from gymnasium.utils.env_checker import check_env
import numpy as np

# Create a NEW environment using the fixed class.
env = make_env(sim, env_cfg)

check_env(env, skip_render_check=True)

obs1, _ = env.reset(seed=42)
obs2, _ = env.reset(seed=42)
assert np.array_equal(obs1, obs2), "Seed reproducibility check failed."

print("Environment checks passed.")
print("Seed reproducibility passed.")
print("Observation shape:", obs2.shape)
print("Number of actions:", env.action_space.n)

env.close()

Environment checks passed.
Seed reproducibility passed.
Observation shape: (24,)
Number of actions: 20


In [20]:
from pathlib import Path
import importlib

from training.config import TrainConfig
import training.train_rl as train_module

# Reload after the environment seed fix.
train_module = importlib.reload(train_module)

assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first."
assert (ARTIFACTS / "response_model.joblib").is_file()
assert (DIRS["synthetic"] / "simulator_assumptions.json").is_file()

train_cfg = TrainConfig(
    mode="smoke",
    device="cpu",
)

print("Training settings:")
print(train_cfg.resolved())
print("Outputs will be saved under:", OUT)

Training settings:
TrainConfig(mode='smoke', seeds=(42,), timesteps=2000, device='cpu', checkpoint_freq=500, eval_episodes=5, algorithms=('dqn', 'ppo', 'rule_based', 'lowest_first'))
Outputs will be saved under: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs


In [21]:
import time

started = time.time()
print("Starting DQN + PPO smoke training and baseline comparison...", flush=True)

results = train_module.run_training_comparison(
    simulator=sim,
    env_config=env_cfg,
    train_config=train_cfg,
    output_root=Path(OUT),
)

print("\nSmoke training completed.")
print("Elapsed minutes:", round((time.time() - started) / 60, 2))
print("Run folder:", results["run_dir"])
print("Validation selection:", results["selection"])

Starting DQN + PPO smoke training and baseline comparison...


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Smoke training completed.
Elapsed minutes: 0.5
Run folder: /content/drive/MyDrive/AdaptiveLearnSE/c4_runs/train_smoke_20261012T031844.758570+0000
Validation selection: {'selected': 'rule_based', 'mean_val_return_by_algorithm': {'dqn': -8.016000018119811, 'ppo': -39.64200009584427, 'rule_based': 5.780000534653668, 'lowest_first': 5.780000534653668}, 'note': 'Selection uses validation only; do not tune on final test.'}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
import pandas as pd

rows = []

for seed, methods in results["seeds"].items():
    for method, payload in methods.items():
        metrics = payload["val"]
        rows.append({
            "seed": seed,
            "method": method,
            "mean_return": metrics["return"]["mean"],
            "simulated_mastery_gain": metrics["mastery_gain_sum"]["mean"],
            "invalid_action_rate": metrics["invalid_action_rate"]["mean"],
            "mean_steps": metrics["time_cost_proxy_steps"]["mean"],
        })

display(pd.DataFrame(rows))

run_dir = Path(results["run_dir"])

print("\nSaved model files:")
for path in sorted(run_dir.rglob("*_final.zip")):
    print(path)

print("\nComparison file exists:",
      (run_dir / "comparison_val.json").is_file())

,seed,method,mean_return,simulated_mastery_gain,invalid_action_rate,mean_steps
0,42,dqn,-8.016000,1.544000,0.296,50.0
1,42,ppo,-39.642000,0.388000,0.828,50.0
2,42,rule_based,5.780001,1.780001,0.000,50.0
3,42,lowest_first,5.780001,1.780001,0.000,50.0



Saved model files:
/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/train_smoke_20261012T031844.758570+0000/seed_42/dqn/dqn_final.zip
/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/train_smoke_20261012T031844.758570+0000/seed_42/ppo/ppo_final.zip

Comparison file exists: True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
